# Closed loop: LUT vs NN
$[\theta_{ref}]$ → $[e=\theta_{ref}-\theta(t-\Delta)]$ → [PID] → $[\tau_{fb}]$

$\tau_{req}=\tau_{fb}+\tau_{ff}$ → LUT or NN → $\vec u$ → MyoSuite/MuJoCo → $\theta$.

$\Delta$ = `DELAY_MS` (SNN of near-nes: 150 ms); $\tau_{ff}$ from the inverse dynamics of the minimum-jerk trajectory (none for the step).

## Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

from src.lut.lut import LUT
from src.nn.nn_validator import NN_validator
from src.nn.nn_model_visualizer import NN_model_container
from src.sim.trajectory import generate_minjerk_trajectory
from src.sim.elbow_simulation import (load_model, simulate_closed_loop, feedforward_torque, save_simulations, simulation_path,
                                  simulation_title, plot_angle, plot_torque, plot_activations, plot_reference,
                                  pid_grid_search)
from src.lut.lut_generator import LUT_generator

## Parameters

In [ ]:
PATH_LUT = "models/luts/W_all/lut_W_all.nc"
PATH_MUJOCO_MODEL = "models/myo_sim/elbow/myoelbow_1dof6muscles.xml"

MODELS_STRUCTURE = ["3L_64U"]  # "2L_32U", "2L_64U", "3L_64U"
ACTIVATION_MODE = ["relu"]
OUTPUT_ACTIVATION_MODE = ["linear", "linear_cos_decay"]

PATH_MODELS = []
for struct in MODELS_STRUCTURE:
    for act in ACTIVATION_MODE:
        for out in OUTPUT_ACTIVATION_MODE:
            PATH_MODELS.append(f'models/nn/W_all/grid_search/W_all_{struct}_{act}_{out}')

TIMESTEP = 0.001  # [s]
P_VALUE = 2

## Upload LUT, NN and MuJoCo model
`NN_validator` loads the Keras model (with `LUT_preprocessing`) and compares it with the LUT; its method `predict(W, angle, torque)` is used in the loop.

In [ ]:
my_lut = LUT(PATH_LUT)
my_lut.load_data()

model = load_model(PATH_MUJOCO_MODEL, timestep=TIMESTEP)

my_lut_generator = LUT_generator(PATH_MUJOCO_MODEL, p_value=P_VALUE, w_values=[0.5])

allocators = {"LUT": my_lut.get_activations,
              "solver": my_lut_generator.get_activations}
allocator_paths = {"LUT": PATH_LUT,
                   "solver": f"{PATH_MUJOCO_MODEL} (online, p={P_VALUE:g})"}

for path in PATH_MODELS:
    container = NN_model_container(path)
    nn = NN_validator(container.path_keras, my_lut)

    allocators[nn.name] = lambda W, a, t, v, nn=nn: nn.predict(W, a, t)
    allocator_paths[nn.name] = container.path_keras

## Simulation
$(W, angle, torque) \Rightarrow u$ from LUT vs NN.

In [ ]:
TRAJECTORY = "minjerk"   # "step"

W_ref = 0.5
init_angle = 20 # [°]
angle_ref = 90 # [°]

K_P = 1.0 #0.039
K_I = 1.0 #0.006
K_D = 0.0 #0.009

DELAY_MS = 0.0#150.0  # [ms] delay of the feedback in the PID (SNN: 150 ms)
FEEDFORWARD = False  # tau_ff from the inverse dynamics of theta_d (only minjerk)

SIM_TIME_STEP = 6.0  # [s] only for step

# phases of the trajectory [ms], defaults of near-nes
TIME_PREP_MS = 650.0
TIME_MOVE_MS = 500.0
TIME_LOCKED_MS = 150.0
TIME_POST_MS = 3000.0 #200.0

if TRAJECTORY == "step":
    traj = None
    tau_ff = None
    reference = angle_ref
    sim_time = SIM_TIME_STEP
    kind = "step"
    traj_params = {}

elif TRAJECTORY == "minjerk":
    traj_params = dict(TIME_PREP_MS=TIME_PREP_MS, TIME_MOVE_MS=TIME_MOVE_MS,
                       TIME_LOCKED_MS=TIME_LOCKED_MS, TIME_POST_MS=TIME_POST_MS)
    traj = generate_minjerk_trajectory(init_angle, angle_ref, resolution_ms=TIMESTEP * 1000,
                                       time_prep_ms=TIME_PREP_MS, time_move_ms=TIME_MOVE_MS,
                                       time_locked_ms=TIME_LOCKED_MS, time_post_ms=TIME_POST_MS,
                                       lut=my_lut)
    reference = traj["theta_d"].to_numpy()
    tau_ff = feedforward_torque(model, traj) if FEEDFORWARD else None
    sim_time = len(reference) * TIMESTEP
    kind = f"minjerk_{TIME_MOVE_MS:g}ms"

else:
    raise ValueError(f"TRAJECTORY: {TRAJECTORY} not valid.")

sims = {}
for name, allocator in allocators.items():
    sims[name] = simulate_closed_loop(model, allocator, W_ref, reference, sim_time,
                                      init_angle=init_angle, K_P=K_P, K_I=K_I, K_D=K_D,
                                      delay_ms=DELAY_MS, tau_ff=tau_ff)

## Save
One folder per experiment and controller:
`simulations/{kind}_{init_angle}_to_{angle_ref}deg/W_{W_ref}_KP_{K_P}_KI_{K_I}_KD_{K_D}/`
with one CSV per allocator (LUT or NN), `angle_ref.csv` and `simulations_metadata.json` with all the parameters.

In [ ]:
SAVE_PATH_SIMS = simulation_path(init_angle, angle_ref, W_ref, K_P, K_I, K_D, kind=kind,
                                 delay_ms=DELAY_MS, ff=FEEDFORWARD)

parameters = dict(
    TRAJECTORY=TRAJECTORY,
    W_ref=W_ref, angle_ref=angle_ref, init_angle=init_angle,
    **traj_params,
    K_P=K_P, K_I=K_I, K_D=K_D,
    DELAY_MS=DELAY_MS,
    FEEDFORWARD=tau_ff is not None,
    SIM_TIME=sim_time, TIMESTEP=TIMESTEP,
    PATH_MUJOCO_MODEL=PATH_MUJOCO_MODEL,
    allocators=allocator_paths,
)

save_simulations(sims, SAVE_PATH_SIMS, parameters, reference=traj)

## Plots

In [ ]:
# SAVE_PATH_PLOTS = "plots/closed_loop_lut_nn"
SAVE_PATH_PLOTS = None

title = simulation_title(init_angle, angle_ref, W_ref, K_P, K_I, K_D, kind=kind,
                         delay_ms=DELAY_MS, ff=FEEDFORWARD)
plot_angle(sims, title=f"Elbow angle ({title})", save_path=SAVE_PATH_PLOTS)
plot_torque(sims, title=f"Torque ({title})", save_path=SAVE_PATH_PLOTS)
plot_activations(sims, title=f"Activations ({title})", save_path=SAVE_PATH_PLOTS)
if traj is not None:
    plot_reference(traj, title=f"Desired angle ({init_angle}°→{angle_ref}°)", save_path=SAVE_PATH_PLOTS)

## PID Tuning

In [ ]:
results = pid_grid_search(model, my_lut_generator.get_activations, W_ref, reference, sim_time,
                          K_P_values=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
                          K_I_values=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
                          K_D_values=[0.0, 0.1, 0.2],
                          init_angle=init_angle, delay_ms=DELAY_MS, tau_ff=tau_ff)
#display(results)
results.head(10)